# 🍳 PostSoma Kitchen · 全量自动化云端 GPU 封面图生产流水线 (FLUX.1-schnell on A100)

本 Notebook 专为 Google Colab **A100 (40GB/80GB)** 或 **T4 GPU** 定制，全量自动化生成 170 道 VisualRecipeV3 食谱封面：
- **100% 物理化与纯英文化**：彻底杜绝中文残留与无用套话，严格按 `[食材形态与刀工] + [特征酱色与烹饪光泽] + [白瓷盛器] + [微距摄影光影]` 规范，单条 35~45 词，完美适配 FLUX 架构。
- **云端断点续跑 (Idempotent)**：自动拉取 Supabase Storage 已有文件与 Drive 清单，已存在的直接跳过，未生成的自动生成并即时上传。
- **零内存泄漏守卫**：每次推断自动执行 `torch.cuda.empty_cache()` 与垃圾回收，稳定支撑全量 170 道流水线无中断跑完。
- **达标自检规范**：中心裁切为 4:3 比例，阶梯自适应压制为 WebP（严格 ≤ 30KB），生成即时落盘至 Supabase `recipe-covers` 桶。

---
## 🚀 极简执行方式：
1. 确认已在 Colab 左侧 **🔑 (Secrets)** 中配置 `SUPABASE_URL` 与 `SUPABASE_SERVICE_KEY`（并打开 Notebook access 开关）；
2. 顶部菜单点击 **【代码执行程序】➔【全部运行 (Run All)】**，流水线将自动接管直至全部完成！

In [ ]:
# [Cell 1] 环境与核心依赖极速安装
!pip install -q "diffusers>=0.30.0" transformers accelerate sentencepiece protobuf supabase pillow

import os
import io
import gc
import json
import time
import math
import base64
import requests
import torch
from PIL import Image
from IPython.display import display, HTML
from supabase import create_client, Client

print("✅ 依赖库安装成功！")
print(f"CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
    print(f"当前硬件: {gpu_name} | 显存总量: {vram_gb} GB")

In [ ]:
# [Cell 2] 挂载 Google Drive & 安全读取 Secrets & 云端桶就绪校验
from google.colab import drive, userdata

# 挂载 Drive 用于持久化 Manifest 清单
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/PostSoma_Kitchen'
os.makedirs(DRIVE_DIR, exist_ok=True)
MANIFEST_PATH = os.path.join(DRIVE_DIR, 'recipe_cover_manifest.json')

# 从 Secrets 安全沙箱中读取凭据
try:
    SUPABASE_URL = userdata.get('SUPABASE_URL').strip().rstrip('/')
    SUPABASE_SERVICE_KEY = userdata.get('SUPABASE_SERVICE_KEY').strip()
    supabase: Client = create_client(SUPABASE_URL, SUPABASE_SERVICE_KEY)
    print(f"✅ 成功连接 Supabase 节点: {SUPABASE_URL}")
except Exception as e:
    raise RuntimeError(f"❌ 读取 Secrets 失败，请检查左侧 🔑 是否配置了 SUPABASE_URL 和 SUPABASE_SERVICE_KEY: {e}")

BUCKET_NAME = 'recipe-covers'
# 检查云端已上传文件清单，用于首层极速断点续跑
CLOUD_EXISTING_FILES = set()
try:
    res = supabase.storage.from_(BUCKET_NAME).list()
    CLOUD_EXISTING_FILES = {f['name'] for f in res if isinstance(f, dict) and 'name' in f}
    print(f"📦 云端 Storage 当前已存在 {len(CLOUD_EXISTING_FILES)} 个封面文件！")
except Exception as e:
    print(f"ℹ️ 无法自动拉取云端文件列表（将依据 Drive Manifest 执行续跑）: {e}")

In [ ]:
# [Cell 3] 加载 FLUX.1-schnell 工业级生成引擎 (支持 A100 全显存直驱与 T4 显存守卫)
from diffusers import FluxPipeline

print("⏳ 正在加载 black-forest-labs/FLUX.1-schnell 模型权重...")
t0 = time.time()

pipe = FluxPipeline.from_pretrained(
    "black-forest-labs/FLUX.1-schnell",
    torch_dtype=torch.bfloat16
)

# 依据显存大小自适应调度
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3) if torch.cuda.is_available() else 0
if total_vram_gb >= 24:
    pipe.to("cuda")
    print(f"🚀 检测到高显存硬件 ({round(total_vram_gb, 1)} GB A100/V100)，启用纯 GPU 全速直驱推断！")
else:
    pipe.enable_model_cpu_offload()
    print(f"✨ 显存为 {round(total_vram_gb, 1)} GB (T4)，启用 CPU Offload 显存守卫！")

print(f"✅ FLUX.1-schnell 加载完成！耗时: {round(time.time() - t0, 1)} 秒")

In [ ]:
# [Cell 4] 170 道食谱 + 8 大通用情境保底图 100% 纯英文物理指令矩阵 (0 中文残留，0 泛化套话)
RECIPES_DATA = [
  {
    "id": "cn-01",
    "title": "🥩 蒜烧五花肉",
    "cleanTitle": "蒜烧五花肉",
    "prompt": "A minimalist plated dish of garlic pork belly. Sliced pork belly with alternating fat-lean layers coated in glistening amber glaze, neatly arranged on a white ceramic platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-02",
    "title": "🥩 猪肉炖粉条",
    "cleanTitle": "猪肉炖粉条",
    "prompt": "A minimalist plated dish of pork stew. Tender braised pork cubes and sweet potato noodles coated in savory brown broth, neatly arranged in a white porcelain bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-03",
    "title": "🥩 私家京酱肉丝",
    "cleanTitle": "私家京酱肉丝",
    "prompt": "A minimalist plated dish of Beijing shredded pork. Pork shreds with crisp scallion threads coated in sweet bean sauce, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-04",
    "title": "🥩 杏鲍菇牛肉粒",
    "cleanTitle": "杏鲍菇牛肉粒",
    "prompt": "A minimalist plated dish of black pepper beef. Beef tenderloin cubes and king oyster mushrooms coated in cracked pepper jus, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-05",
    "title": "🥩 番茄炖牛腩",
    "cleanTitle": "番茄炖牛腩",
    "prompt": "A minimalist plated dish of tomato beef brisket. Seared beef brisket cubes showing rich grain coated in thick tomato gravy, neatly arranged in a white porcelain casserole. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-06",
    "title": "🥩 金针肥牛",
    "cleanTitle": "金针肥牛",
    "prompt": "A minimalist plated dish of enoki beef rolls. Marbled beef rolls wrapping enoki mushrooms coated in spicy garlic sauce, neatly arranged on a white porcelain platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-07",
    "title": "🥩 白萝卜羊肉卷",
    "cleanTitle": "白萝卜羊肉卷",
    "prompt": "A minimalist plated dish of steamed lamb rolls. Lamb rolls wrapped inside translucent daikon ribbons coated in fragrant clear broth, neatly arranged on a white porcelain platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-08",
    "title": "🥩 羊肉炖胡萝卜",
    "cleanTitle": "羊肉炖胡萝卜",
    "prompt": "A minimalist plated dish of braised lamb stew. Slow-cooked lamb chunks and orange carrot wedges coated in clear golden broth, neatly arranged in a white porcelain tureen. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-09",
    "title": "🥩 葱爆羊肉",
    "cleanTitle": "葱爆羊肉",
    "prompt": "A minimalist plated dish of wok scallion lamb. Flash-seared sliced lamb and charred scallion batons coated in aromatic sesame glaze, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-10",
    "title": "🍗 板栗烧鸡",
    "cleanTitle": "板栗烧鸡",
    "prompt": "A minimalist plated dish of chestnut braised chicken. Bone-in chicken chunks and sweet chestnuts coated in rich savory glaze, neatly arranged in a white ceramic bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-11",
    "title": "🍗 宫保鸡丁",
    "cleanTitle": "宫保鸡丁",
    "prompt": "A minimalist plated dish of Kung Pao chicken. Diced chicken thighs and crunchy roasted peanuts coated in glossy sweet-sour glaze, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-12",
    "title": "🍗 板栗鸡丁",
    "cleanTitle": "板栗鸡丁",
    "prompt": "A minimalist plated dish of sautéed chestnut chicken. Diced chicken breast and golden roasted chestnuts coated in light savory glaze, neatly arranged on a white porcelain dish. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-13",
    "title": "🥚 豆渣蒸蛋",
    "cleanTitle": "豆渣蒸蛋",
    "prompt": "A minimalist plated dish of steamed egg custard. Silky egg custard blended with soybean okara coated in toasted sesame oil, neatly arranged in a white porcelain bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-14",
    "title": "🥕 粉蒸胡萝卜丝",
    "cleanTitle": "粉蒸胡萝卜丝",
    "prompt": "A minimalist plated dish of steamed carrots. Carrot ribbons dusted with golden cornmeal crust coated in fragrant scallion oil, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-15",
    "title": "🥩 胡萝卜牛腩煲",
    "cleanTitle": "胡萝卜牛腩煲",
    "prompt": "A minimalist plated dish of carrot beef. Beef brisket cubes and sweet carrots coated in dark savory gravy, neatly arranged in a white ceramic casserole. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 48
  },
  {
    "id": "cn-16",
    "title": "🥩 私房少油鱼香肉丝",
    "cleanTitle": "私房少油鱼香肉丝",
    "prompt": "A minimalist plated dish of Yuxiang pork. Tender pork strips and crisp wood-ear mushrooms coated in sweet-tangy chili glaze, neatly arranged on a white ceramic platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-17",
    "title": "🍆 蒜蓉蒸茄子",
    "cleanTitle": "蒜蓉蒸茄子",
    "prompt": "A minimalist plated dish of garlic eggplant. Purple eggplant batons topped with golden garlic coated in light soy drizzle, neatly arranged on a white porcelain dish. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-18",
    "title": "🦐 鲜虾茄子煲",
    "cleanTitle": "鲜虾茄子煲",
    "prompt": "A minimalist plated dish of prawn eggplant. Pink prawns and tender eggplant batons coated in fragrant garlic oyster sauce, neatly arranged in a white ceramic bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-19",
    "title": "🍳 地三鲜",
    "cleanTitle": "地三鲜",
    "prompt": "A minimalist plated dish of triple harvest. Golden potato wedges and tender eggplant coated in glossy garlic soy glaze, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-20",
    "title": "🍅 秘制番茄酱",
    "cleanTitle": "秘制番茄酱",
    "prompt": "A minimalist plated dish of tomato compote. Simmered vine tomato compote showing pulp coated in natural juices, neatly arranged in a white ceramic ramekin. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 47
  },
  {
    "id": "cn-21",
    "title": "🧈 减脂番茄豆腐羹",
    "cleanTitle": "减脂番茄豆腐羹",
    "prompt": "A minimalist plated dish of tomato tofu. Silken white tofu cubes and fluffy egg ribbons coated in red tomato broth, neatly arranged in a white porcelain bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-22",
    "title": "🍗 番茄炒鸡蛋",
    "cleanTitle": "番茄炒鸡蛋",
    "prompt": "A minimalist plated dish of tomato eggs. Fluffy egg curds and ripe tomato wedges coated in sweet-savory tomato glaze, neatly arranged on a white ceramic platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-23",
    "title": "🥩 萝卜丝蒸牛肉",
    "cleanTitle": "萝卜丝蒸牛肉",
    "prompt": "A minimalist plated dish of daikon beef. Marinated beef strips laid over daikon ribbons coated in light soy jus, neatly arranged on a white porcelain dish. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-24",
    "title": "🥩 冬瓜薏米排骨汤",
    "cleanTitle": "冬瓜薏米排骨汤",
    "prompt": "A minimalist plated dish of melon rib soup. Pork ribs and translucent winter melon cubes coated in clear fragrant broth, neatly arranged in a white porcelain tureen. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-25",
    "title": "🎃 红枣百合蒸南瓜",
    "cleanTitle": "红枣百合蒸南瓜",
    "prompt": "A minimalist plated dish of steamed pumpkin. Sweet pumpkin slices and fresh lily bulbs coated in blossom honey glaze, neatly arranged on a white porcelain plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-26",
    "title": "🥒 黄瓜炒甜椒",
    "cleanTitle": "黄瓜炒甜椒",
    "prompt": "A minimalist plated dish of cucumber peppers. Emerald cucumber ribbons and sweet bell peppers coated in light glistening oil, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-27",
    "title": "🥒 苦瓜冬菇骨汤",
    "cleanTitle": "苦瓜冬菇骨汤",
    "prompt": "A minimalist plated dish of bitter gourd soup. Tender pork ribs and bitter gourd rings coated in clear savory broth, neatly arranged in a white porcelain bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-28",
    "title": "🥚 苦瓜蒸咸蛋",
    "cleanTitle": "苦瓜蒸咸蛋",
    "prompt": "A minimalist plated dish of steamed bitter gourd. Bitter gourd rings with salted egg custard coated in savory steam jus, neatly arranged on a white porcelain dish. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-29",
    "title": "🥩 苦瓜肉片",
    "cleanTitle": "苦瓜肉片",
    "prompt": "A minimalist plated dish of bitter gourd pork. Sliced pork and crisp bitter gourd ribbons coated in garlic oyster glaze, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-30",
    "title": "🥒 翡翠丝瓜卷",
    "cleanTitle": "翡翠丝瓜卷",
    "prompt": "A minimalist plated dish of emerald luffa rolls. Steamed fish paste wrapped inside luffa sheets coated in light seafood jus, neatly arranged on a white porcelain platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-31",
    "title": "🥬 韭菜豆芽菜松",
    "cleanTitle": "韭菜豆芽菜松",
    "prompt": "A minimalist plated dish of chive bean sprouts. Crisp bean sprouts and fragrant garlic chives coated in sesame oil sheen, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-32",
    "title": "♨️ 翡翠白玉卷",
    "cleanTitle": "翡翠白玉卷",
    "prompt": "A minimalist plated dish of steamed cabbage rolls. Cabbage leaf rolls wrapping seasoned tofu coated in clear fragrant broth, neatly arranged on a white porcelain dish. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-33",
    "title": "🥩 白菜粉丝猪骨汤",
    "cleanTitle": "白菜粉丝猪骨汤",
    "prompt": "A minimalist plated dish of cabbage pork broth. Tender pork bones and Chinese cabbage hearts coated in milky white broth, neatly arranged in a white porcelain tureen. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-34",
    "title": "🥬 草菇炒白菜",
    "cleanTitle": "草菇炒白菜",
    "prompt": "A minimalist plated dish of straw mushroom greens. Plump straw mushrooms and Chinese cabbage hearts coated in garlic oyster sauce, neatly arranged on a white ceramic dish. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-35",
    "title": "🥩 洋葱炒猪肝",
    "cleanTitle": "洋葱炒猪肝",
    "prompt": "A minimalist plated dish of onion pork liver. Sliced pork liver and caramelized onion ribbons coated in savory soy glaze, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-36",
    "title": "🥬 飘香手撕圆白菜",
    "cleanTitle": "飘香手撕圆白菜",
    "prompt": "A minimalist plated dish of hand-torn cabbage. Crisp green cabbage leaves and dried chili pods coated in chili vinegar glaze, neatly arranged on a white ceramic platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-37",
    "title": "🥬 海米油菜",
    "cleanTitle": "海米油菜",
    "prompt": "A minimalist plated dish of baby bok choy. Bok choy hearts and dried baby shrimp coated in light garlic broth, neatly arranged on a white porcelain platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-38",
    "title": "🦐 蚝油生菜",
    "cleanTitle": "蚝油生菜",
    "prompt": "A minimalist plated dish of oyster sauce lettuce. Romaine lettuce leaves and garlic threads coated in savory oyster sauce, neatly arranged on a white porcelain plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-39",
    "title": "🥩 芹菜腊肉丁",
    "cleanTitle": "芹菜腊肉丁",
    "prompt": "A minimalist plated dish of celery cured bacon. Crisp celery batons and cured bacon cubes coated in aromatic glistening oil, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-40",
    "title": "🥬 白灼芥蓝",
    "cleanTitle": "白灼芥蓝",
    "prompt": "A minimalist plated dish of poached gai lan. Chinese broccoli stalks and sliced ginger threads coated in sizzling soy sauce, neatly arranged on a white porcelain platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-41",
    "title": "🥚 蛋黄苋菜",
    "cleanTitle": "蛋黄苋菜",
    "prompt": "A minimalist plated dish of salted yolk amaranth. Green amaranth leaves and garlic cloves coated in salted egg yolk sauce, neatly arranged in a white porcelain bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-42",
    "title": "🦐 虾仁蒸西蓝花",
    "cleanTitle": "虾仁蒸西蓝花",
    "prompt": "A minimalist plated dish of shrimp broccoli. Pink shrimp and emerald broccoli florets coated in egg white glaze, neatly arranged on a white porcelain plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 48
  },
  {
    "id": "cn-43",
    "title": "🥒 莴笋聚会",
    "cleanTitle": "莴笋聚会",
    "prompt": "A minimalist plated dish of celtuce medley. Translucent celtuce batons and winter bamboo shoots coated in sesame garlic oil, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-44",
    "title": "🥩 莴笋烧肉",
    "cleanTitle": "莴笋烧肉",
    "prompt": "A minimalist plated dish of pork celtuce. Braised pork cubes and jade celtuce batons coated in amber brown gravy, neatly arranged in a white ceramic casserole. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-45",
    "title": "🥒 姜汁莴笋",
    "cleanTitle": "姜汁莴笋",
    "prompt": "A minimalist plated dish of ginger celtuce. Julienned celtuce ribbons and sweet red peppers coated in tangy ginger dressing, neatly arranged on a white porcelain dish. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-46",
    "title": "🍗 西葫芦炒鸡蛋",
    "cleanTitle": "西葫芦炒鸡蛋",
    "prompt": "A minimalist plated dish of zucchini scrambled eggs. Fluffy egg curds and green zucchini slices coated in light savory oil, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-47",
    "title": "🍄 香菇藕丸",
    "cleanTitle": "香菇藕丸",
    "prompt": "A minimalist plated dish of lotus root meatballs. Pork meatballs with lotus root and shiitake coated in savory oyster jus, neatly arranged on a white porcelain dish. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-48",
    "title": "🍄 金针菇培根卷",
    "cleanTitle": "金针菇培根卷",
    "prompt": "A minimalist plated dish of enoki bacon rolls. Crispy bacon strips wrapping enoki mushrooms coated in sweet soy glaze, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-49",
    "title": "🍳 醋熘素什锦",
    "cleanTitle": "醋熘素什锦",
    "prompt": "A minimalist plated dish of sweet sour vegetables. Carrot slices, Chinese yam, and wood-ear mushrooms coated in glossy vinegar sauce, neatly arranged on a white ceramic platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-50",
    "title": "🍄 银耳百合雪梨汤",
    "cleanTitle": "银耳百合雪梨汤",
    "prompt": "A minimalist plated dish of snow fungus soup. Snow fungus and poached sweet pear wedges coated in clear sweet syrup, neatly arranged in a white porcelain bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-51",
    "title": "🥔 荷香小米蒸红薯",
    "cleanTitle": "荷香小米蒸红薯",
    "prompt": "A minimalist plated dish of lotus sweet potato. Sweet potato cubes and steamed yellow millet coated in fragrant honey glaze, neatly arranged in a white porcelain bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-52",
    "title": "🥔 山药寿司",
    "cleanTitle": "山药寿司",
    "prompt": "A minimalist plated dish of yam sushi. Steamed white yam rolls wrapped in nori seaweed coated in toasted sesame oil, neatly arranged on a white ceramic platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-53",
    "title": "🍲 关东煮",
    "cleanTitle": "关东煮",
    "prompt": "A minimalist plated dish of vegetable oden. Simmered daikon radish cubes and shiitake caps coated in clear dashi broth, neatly arranged in a white porcelain bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-54",
    "title": "🥔 奶香土豆泥",
    "cleanTitle": "奶香土豆泥",
    "prompt": "A minimalist plated dish of mashed potatoes. Whipped potato puree blended with rich cream coated in melted butter, neatly arranged in a white porcelain bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 48
  },
  {
    "id": "cn-55",
    "title": "🐟 泡椒蒸鱼块",
    "cleanTitle": "泡椒蒸鱼块",
    "prompt": "A minimalist plated dish of pickled chili fish. Tender fish steaks with red pickled chilies coated in savory soy broth, neatly arranged on a white porcelain dish. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-56",
    "title": "🐟 红烧鱼豆腐",
    "cleanTitle": "红烧鱼豆腐",
    "prompt": "A minimalist plated dish of braised fish tofu. Golden fish fillets and soft tofu cubes coated in savory brown gravy, neatly arranged in a white ceramic bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-57",
    "title": "🐟 奶白鲫鱼汤",
    "cleanTitle": "奶白鲫鱼汤",
    "prompt": "A minimalist plated dish of crucian carp soup. Fresh crucian carp and soft tofu cubes coated in milky white broth, neatly arranged in a white porcelain tureen. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-58",
    "title": "🦐 蒜蓉基围虾",
    "cleanTitle": "蒜蓉基围虾",
    "prompt": "A minimalist plated dish of garlic prawns. Butterfly pink prawns topped with minced garlic coated in light scallion oil, neatly arranged on a white porcelain platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-59",
    "title": "🍅 番茄炖牡蛎",
    "cleanTitle": "番茄炖牡蛎",
    "prompt": "A minimalist plated dish of tomato oysters. Fresh plump oysters and ripe vine tomato chunks coated in rich tomato sauce, neatly arranged in a white porcelain casserole. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-60",
    "title": "🦐 蒜蓉粉丝蒸扇贝",
    "cleanTitle": "蒜蓉粉丝蒸扇贝",
    "prompt": "A minimalist plated dish of garlic scallops. Sea scallops with glass noodles and minced garlic coated in light soy dressing, neatly arranged on a white porcelain platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-61",
    "title": "🦐 姜丝蒸蟹",
    "cleanTitle": "姜丝蒸蟹",
    "prompt": "A minimalist plated dish of steamed crab. Steamed whole crab with golden roe and ginger coated in rice wine sauce, neatly arranged on a white porcelain platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-62",
    "title": "🥩 家常榨菜蒸牛肉",
    "cleanTitle": "家常榨菜蒸牛肉",
    "prompt": "A minimalist plated dish of pickled mustard beef. Marinated beef slices and pickled mustard greens coated in savory steam jus, neatly arranged on a white porcelain platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-63",
    "title": "🍲 茄汁黄豆",
    "cleanTitle": "茄汁黄豆",
    "prompt": "A minimalist plated dish of tomato soybeans. Plump simmered yellow soybeans and tomato chunks coated in rich tomato gravy, neatly arranged in a white ceramic bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-64",
    "title": "🥕 胡萝卜炒木耳",
    "cleanTitle": "胡萝卜炒木耳",
    "prompt": "A minimalist plated dish of carrot wood-ear stir-fry. Sweet carrot slices and black wood-ear mushrooms coated in glistening garlic oil, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-65",
    "title": "🥬 蒜蓉油淋生菜",
    "cleanTitle": "蒜蓉油淋生菜",
    "prompt": "A minimalist plated dish of sizzling garlic lettuce. Blanched romaine lettuce leaves and golden garlic coated in sizzling soy sauce, neatly arranged on a white porcelain platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-66",
    "title": "🥚 菠菜蒸蛋",
    "cleanTitle": "菠菜蒸蛋",
    "prompt": "A minimalist plated dish of spinach egg custard. Silky egg custard with vibrant spinach leaves coated in toasted sesame oil, neatly arranged in a white porcelain bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-67",
    "title": "🧈 香菇什锦豆腐",
    "cleanTitle": "香菇什锦豆腐",
    "prompt": "A minimalist plated dish of mushroom tofu. Braised firm tofu cubes and shiitake mushrooms coated in brown oyster gravy, neatly arranged in a white ceramic dish. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-68",
    "title": "🐟 醋烹带鱼",
    "cleanTitle": "醋烹带鱼",
    "prompt": "A minimalist plated dish of crispy ribbonfish. Pan-seared ribbonfish steaks and ginger threads coated in black vinegar glaze, neatly arranged on a white ceramic platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 48
  },
  {
    "id": "cn-69",
    "title": "🍗 板栗蒸土鸡",
    "cleanTitle": "板栗蒸土鸡",
    "prompt": "A minimalist plated dish of chestnut chicken. Bone-in chicken chunks and sweet golden chestnuts coated in savory chicken broth, neatly arranged on a white porcelain dish. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-70",
    "title": "🥩 白菜羊肉丸子汤",
    "cleanTitle": "白菜羊肉丸子汤",
    "prompt": "A minimalist plated dish of lamb meatball soup. Tender minced lamb meatballs and cabbage hearts coated in warming herbal broth, neatly arranged in a white porcelain tureen. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-71",
    "title": "🍗 鸡丁核桃仁",
    "cleanTitle": "鸡丁核桃仁",
    "prompt": "A minimalist plated dish of walnut chicken. Diced chicken breast cubes and roasted walnuts coated in light savory glaze, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-72",
    "title": "🐟 清蒸鲢鱼",
    "cleanTitle": "清蒸鲢鱼",
    "prompt": "A minimalist plated dish of steamed silver carp. Silver carp fillet and fresh cilantro sprigs coated in light soy jus, neatly arranged on a white porcelain platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-73",
    "title": "🥩 猪肉黄豆炖豆腐",
    "cleanTitle": "猪肉黄豆炖豆腐",
    "prompt": "A minimalist plated dish of pork soybean stew. Pork belly, yellow soybeans, and soft tofu coated in savory soy gravy, neatly arranged in a white ceramic casserole. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-74",
    "title": "🍳 清炒扁豆丝",
    "cleanTitle": "清炒扁豆丝",
    "prompt": "A minimalist plated dish of sautéed flat beans. Flat bean strips and mild red chili coated in glistening garlic oil, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-75",
    "title": "🥬 如意白菜卷",
    "cleanTitle": "如意白菜卷",
    "prompt": "A minimalist plated dish of auspicious cabbage rolls. Steamed cabbage rolls filled with seasoned pork coated in clear egg broth, neatly arranged on a white porcelain platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-76",
    "title": "♨️ 双耳羹",
    "cleanTitle": "双耳羹",
    "prompt": "A minimalist plated dish of two fungus soup. Snow fungus and crunchy black wood-ear mushrooms coated in clear sweet syrup, neatly arranged in a white porcelain bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-77",
    "title": "🥩 胡萝卜雪梨炖瘦肉",
    "cleanTitle": "胡萝卜雪梨炖瘦肉",
    "prompt": "A minimalist plated dish of carrot pear pork soup. Lean pork, carrots, and sweet pear wedges coated in fragrant broth, neatly arranged in a white porcelain tureen. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-78",
    "title": "🥩 荸荠炒肉丝",
    "cleanTitle": "荸荠炒肉丝",
    "prompt": "A minimalist plated dish of water chestnut pork. Pork loin strips and crisp water chestnuts coated in savory oyster sauce, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-79",
    "title": "🍳 多味冬瓜",
    "cleanTitle": "多味冬瓜",
    "prompt": "A minimalist plated dish of savory winter melon. Winter melon slices with dried shrimp coated in fragrant mushroom jus, neatly arranged on a white porcelain dish. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-80",
    "title": "♨️ 清蒸竹笋",
    "cleanTitle": "清蒸竹笋",
    "prompt": "A minimalist plated dish of steamed bamboo shoots. Bamboo shoot spears and delicate egg ribbons coated in clear chicken broth, neatly arranged on a white porcelain platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-81",
    "title": "🦐 海带炖豆腐",
    "cleanTitle": "海带炖豆腐",
    "prompt": "A minimalist plated dish of kelp stewed tofu. Firm tofu cubes and shredded kelp ribbons coated in ginger soy broth, neatly arranged in a white ceramic bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-82",
    "title": "🥬 豆干炒菠菜",
    "cleanTitle": "豆干炒菠菜",
    "prompt": "A minimalist plated dish of tofu spinach stir-fry. Pressed tofu shreds and baby spinach leaves coated in garlic sesame oil, neatly arranged on a white ceramic platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-83",
    "title": "🍄 西蓝花蒸蘑菇",
    "cleanTitle": "西蓝花蒸蘑菇",
    "prompt": "A minimalist plated dish of steamed broccoli mushrooms. Broccoli florets and button mushroom halves coated in garlic soy dressing, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-84",
    "title": "🥩 沙茶牛肉",
    "cleanTitle": "沙茶牛肉",
    "prompt": "A minimalist plated dish of Shacha beef. Sliced beef tenderloin and green bell pepper strips coated in aromatic Shacha glaze, neatly arranged on a white ceramic platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-85",
    "title": "🥬 粉蒸芹菜叶",
    "cleanTitle": "粉蒸芹菜叶",
    "prompt": "A minimalist plated dish of steamed celery leaves. Tender celery leaves dusted in seasoned flour coated in garlic sesame oil, neatly arranged on a white porcelain platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-86",
    "title": "🥬 清炒菠菜",
    "cleanTitle": "清炒菠菜",
    "prompt": "A minimalist plated dish of sautéed garlic spinach. Emerald spinach leaves and golden minced garlic coated in light vegetable oil, neatly arranged on a white ceramic platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-87",
    "title": "🥚 洋葱蒸蛋",
    "cleanTitle": "洋葱蒸蛋",
    "prompt": "A minimalist plated dish of onion egg custard. Silky egg custard with sweet red onions coated in light soy drizzle, neatly arranged in a white porcelain bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-88",
    "title": "🐟 虾仁鱼片炖豆腐",
    "cleanTitle": "虾仁鱼片炖豆腐",
    "prompt": "A minimalist plated dish of seafood tofu stew. Pink shrimp, fish fillets, and silken tofu coated in clear seafood broth, neatly arranged in a white porcelain bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-89",
    "title": "🥩 牛奶蒸蛋",
    "cleanTitle": "牛奶蒸蛋",
    "prompt": "A minimalist plated dish of milk egg custard. Milk egg custard topped with pink shrimp coated in light sesame oil, neatly arranged in a white porcelain bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-90",
    "title": "🍗 海带炖鸭汤",
    "cleanTitle": "海带炖鸭汤",
    "prompt": "A minimalist plated dish of kelp duck soup. Tender duck pieces and shredded kelp ribbons coated in savory clear broth, neatly arranged in a white porcelain tureen. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-91",
    "title": "🥔 姜汁红薯条",
    "cleanTitle": "姜汁红薯条",
    "prompt": "A minimalist plated dish of ginger sweet potato. Roasted sweet potato batons and carrot sticks coated in candied ginger glaze, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-92",
    "title": "🧈 番茄炖豆腐",
    "cleanTitle": "番茄炖豆腐",
    "prompt": "A minimalist plated dish of tomato tofu. White tofu cubes and ripe tomato wedges coated in sweet-savory red gravy, neatly arranged in a white porcelain bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-93",
    "title": "🥕 纳豆炖萝卜",
    "cleanTitle": "纳豆炖萝卜",
    "prompt": "A minimalist plated dish of radish natto. Simmered daikon radish rounds and natto beans coated in savory dashi broth, neatly arranged in a white ceramic bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-94",
    "title": "🥒 毛豆烧丝瓜",
    "cleanTitle": "毛豆烧丝瓜",
    "prompt": "A minimalist plated dish of luffa edamame. Emerald luffa gourd batons and green edamame beans coated in garlic broth, neatly arranged in a white porcelain bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-95",
    "title": "🥬 蒜蒸白菜",
    "cleanTitle": "蒜蒸白菜",
    "prompt": "A minimalist plated dish of garlic steamed cabbage. Chinese cabbage hearts layered with minced garlic coated in light soy drizzle, neatly arranged on a white ceramic platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-96",
    "title": "🥔 香蕉土豆泥",
    "cleanTitle": "香蕉土豆泥",
    "prompt": "A minimalist plated dish of banana potato mash. Whipped potato puree blended with ripe banana pulp coated in sweet cream, neatly arranged in a white porcelain bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-97",
    "title": "🥒 山药木耳炒莴笋",
    "cleanTitle": "山药木耳炒莴笋",
    "prompt": "A minimalist plated dish of celtuce yam stir-fry. Celtuce slices, yam, and wood-ear mushrooms coated in light garlic oil, neatly arranged on a white porcelain platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-98",
    "title": "🍄 银耳炖木瓜",
    "cleanTitle": "银耳炖木瓜",
    "prompt": "A minimalist plated dish of papaya snow fungus soup. Orange papaya and translucent snow fungus coated in sweet almond syrup, neatly arranged in a white porcelain bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-99",
    "title": "🎃 南瓜羹",
    "cleanTitle": "南瓜羹",
    "prompt": "A minimalist plated dish of pumpkin puree. Golden pumpkin soup puree and roasted seeds coated in sweet coconut cream, neatly arranged in a white ceramic bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-100",
    "title": "🥩 羊肉炖白萝卜",
    "cleanTitle": "羊肉炖白萝卜",
    "prompt": "A minimalist plated dish of lamb daikon stew. Tender lamb chunks and daikon radish cubes coated in clear fragrant broth, neatly arranged in a white porcelain tureen. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-101",
    "title": "🍳 醋熘绿豆芽",
    "cleanTitle": "醋熘绿豆芽",
    "prompt": "A minimalist plated dish of vinegar bean sprouts. Crisp mung bean sprouts and red chili coated in Sichuan pepper vinegar, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-102",
    "title": "🥔 淮山药泥",
    "cleanTitle": "淮山药泥",
    "prompt": "A minimalist plated dish of Chinese yam paste. Molded yam puree and red bean paste coated in tart hawthorn glaze, neatly arranged on a white porcelain plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-103",
    "title": "🥕 蜂蜜白萝卜盅",
    "cleanTitle": "蜂蜜白萝卜盅",
    "prompt": "A minimalist plated dish of honey daikon cup. Steamed white daikon cup and wolfberries coated in wild blossom honey, neatly arranged on a white porcelain dish. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-104",
    "title": "🍗 西洋参板栗蒸乌鸡",
    "cleanTitle": "西洋参板栗蒸乌鸡",
    "prompt": "A minimalist plated dish of ginseng black chicken. Silkie black chicken pieces and sweet chestnuts coated in ginseng herbal broth, neatly arranged in a white ceramic tureen. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-105",
    "title": "🥩 丝瓜猪肝瘦肉汤",
    "cleanTitle": "丝瓜猪肝瘦肉汤",
    "prompt": "A minimalist plated dish of pork liver soup. Pork liver, lean pork, and tender luffa coated in clear ginger broth, neatly arranged in a white porcelain tureen. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-106",
    "title": "🥩 当归生姜羊肉汤",
    "cleanTitle": "当归生姜羊肉汤",
    "prompt": "A minimalist plated dish of Angelica lamb soup. Lamb slices and fresh ginger root coated in warming Angelica broth, neatly arranged in a white porcelain bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-107",
    "title": "🍲 荔枝桂圆山楂汤",
    "cleanTitle": "荔枝桂圆山楂汤",
    "prompt": "A minimalist plated dish of lychee longan soup. Peeled lychees, dried red longan, and hawthorn coated in golden sweet syrup, neatly arranged in a white porcelain bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-108",
    "title": "🥬 鲜蒸白菜心",
    "cleanTitle": "鲜蒸白菜心",
    "prompt": "A minimalist plated dish of steamed cabbage hearts. Baby cabbage hearts, wood-ear, and baby shrimp coated in light seafood broth, neatly arranged on a white porcelain platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-109",
    "title": "🧈 菠萝豆腐",
    "cleanTitle": "菠萝豆腐",
    "prompt": "A minimalist plated dish of pineapple tofu. Seared firm tofu blocks and yellow pineapple chunks coated in sweet-tangy glaze, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-110",
    "title": "🥩 羊肉苹果汤",
    "cleanTitle": "羊肉苹果汤",
    "prompt": "A minimalist plated dish of lamb apple stew. Lamb chunks, sweet apples, and green peas coated in savory broth, neatly arranged in a white ceramic tureen. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-111",
    "title": "🥩 莲藕排骨汤",
    "cleanTitle": "莲藕排骨汤",
    "prompt": "A minimalist plated dish of lotus rib soup. Pork ribs and sliced lotus root rounds coated in fragrant savory broth, neatly arranged in a white porcelain tureen. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-112",
    "title": "🥒 梅干菜蒸苦瓜",
    "cleanTitle": "梅干菜蒸苦瓜",
    "prompt": "A minimalist plated dish of preserved greens bitter gourd. Bitter gourd and preserved mustard greens coated in savory steam jus, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-113",
    "title": "🍳 海米冬瓜",
    "cleanTitle": "海米冬瓜",
    "prompt": "A minimalist plated dish of shrimp winter melon. Winter melon and golden dried baby shrimp coated in clear savory broth, neatly arranged in a white porcelain bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-114",
    "title": "🍅 番茄炒草菇",
    "cleanTitle": "番茄炒草菇",
    "prompt": "A minimalist plated dish of tomato straw mushrooms. Straw mushrooms and ripe red tomato wedges coated in rich tomato sauce, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-115",
    "title": "🐟 豉汁蒸盘龙白鳝",
    "cleanTitle": "豉汁蒸盘龙白鳝",
    "prompt": "A minimalist plated dish of black bean eel. White eel coiled with fermented black beans coated in garlic soy sauce, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-116",
    "title": "🧈 泥鳅豆腐汤",
    "cleanTitle": "泥鳅豆腐汤",
    "prompt": "A minimalist plated dish of loach fish soup. Freshwater loach and soft white tofu cubes coated in milky fish broth, neatly arranged in a white porcelain tureen. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-117",
    "title": "🦐 蚕豆炒虾仁",
    "cleanTitle": "蚕豆炒虾仁",
    "prompt": "A minimalist plated dish of broad bean shrimp. Pink prawns and tender broad beans coated in light egg white glaze, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-118",
    "title": "🍲 酒炖蛤蜊",
    "cleanTitle": "酒炖蛤蜊",
    "prompt": "A minimalist plated dish of wine steamed clams. Fresh clams in shell with scallions coated in aromatic wine broth, neatly arranged in a white porcelain bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-119",
    "title": "🧈 私家山药炖豆腐",
    "cleanTitle": "私家山药炖豆腐",
    "prompt": "A minimalist plated dish of yam braised tofu. White tofu blocks and Chinese yam batons coated in savory brown gravy, neatly arranged in a white ceramic bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-120",
    "title": "🥩 爆炒羊肝",
    "cleanTitle": "爆炒羊肝",
    "prompt": "A minimalist plated dish of sautéed lamb liver. Sliced lamb liver and bell peppers coated in savory garlic soy glaze, neatly arranged on a white ceramic platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-121",
    "title": "🥩 农家粉蒸牛肉",
    "cleanTitle": "农家粉蒸牛肉",
    "prompt": "A minimalist plated dish of spiced steamed beef. Beef strips in seasoned toasted rice flour crust coated in cilantro oil, neatly arranged on a white porcelain platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-122",
    "title": "🍗 番茄罗勒炖鸡胸",
    "cleanTitle": "番茄罗勒炖鸡胸",
    "prompt": "A minimalist plated dish of tomato basil chicken. Diced chicken breast and sweet tomato sauce coated in fragrant basil jus, neatly arranged in a white porcelain bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-123",
    "title": "🍗 栗子杜仲鸡爪汤",
    "cleanTitle": "栗子杜仲鸡爪汤",
    "prompt": "A minimalist plated dish of chicken feet soup. Chicken feet and roasted sweet chestnuts coated in warming herbal broth, neatly arranged in a white porcelain tureen. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-124",
    "title": "🥩 银耳木瓜排骨汤",
    "cleanTitle": "银耳木瓜排骨汤",
    "prompt": "A minimalist plated dish of papaya rib soup. Pork ribs, orange papaya, and snow fungus coated in delicate savory broth, neatly arranged in a white porcelain tureen. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-125",
    "title": "🥚 鸽蛋西蓝花",
    "cleanTitle": "鸽蛋西蓝花",
    "prompt": "A minimalist plated dish of pigeon eggs broccoli. Boiled pigeon eggs and broccoli florets coated in clear chicken glaze, neatly arranged on a white porcelain platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-126",
    "title": "🍲 蛤蜊冬瓜汤",
    "cleanTitle": "蛤蜊冬瓜汤",
    "prompt": "A minimalist plated dish of clam melon soup. Fresh clam meat and winter melon cubes coated in clear fragrant broth, neatly arranged in a white porcelain bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-127",
    "title": "🥚 香菇蒸蛋",
    "cleanTitle": "香菇蒸蛋",
    "prompt": "A minimalist plated dish of shiitake steamed egg. Egg custard topped with diced shiitake coated in sesame soy dressing, neatly arranged in a white porcelain bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-128",
    "title": "🍲 时蔬乱炖",
    "cleanTitle": "时蔬乱炖",
    "prompt": "A minimalist plated dish of country rib stew. Pork ribs, green beans, and potato cubes coated in savory brown gravy, neatly arranged in a white ceramic casserole. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-129",
    "title": "🥩 豆豉牛肉",
    "cleanTitle": "豆豉牛肉",
    "prompt": "A minimalist plated dish of black bean beef. Sliced beef and green bell pepper strips coated in black bean sauce, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-130",
    "title": "🍄 香菇菜花",
    "cleanTitle": "香菇菜花",
    "prompt": "A minimalist plated dish of cauliflower stir-fry. White cauliflower florets and shiitake caps coated in savory garlic soy sauce, neatly arranged on a white ceramic dish. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-131",
    "title": "🥩 肉炒胡萝卜丝",
    "cleanTitle": "肉炒胡萝卜丝",
    "prompt": "A minimalist plated dish of carrot pork. Shredded pork loin strips and julienned carrot ribbons coated in light savory oil, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-132",
    "title": "♨️ 清蒸牡蛎",
    "cleanTitle": "清蒸牡蛎",
    "prompt": "A minimalist plated dish of steamed oysters. Fresh oysters on half shells with shredded ginger coated in scallion soy dressing, neatly arranged on a white ceramic platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-133",
    "title": "🐟 木瓜鲫鱼汤",
    "cleanTitle": "木瓜鲫鱼汤",
    "prompt": "A minimalist plated dish of papaya fish soup. Crucian carp and sweet papaya wedges coated in milky white fish broth, neatly arranged in a white porcelain tureen. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-134",
    "title": "🥚 咸蛋黄炒南瓜",
    "cleanTitle": "咸蛋黄炒南瓜",
    "prompt": "A minimalist plated dish of salted yolk pumpkin. Roasted sweet pumpkin batons coated in salted duck egg crust, neatly arranged on a white porcelain platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 48
  },
  {
    "id": "cn-135",
    "title": "🐟 清蒸鳕鱼",
    "cleanTitle": "清蒸鳕鱼",
    "prompt": "A minimalist plated dish of steamed cod. Cod fish fillet with ginger and scallions coated in sizzling sesame soy sauce, neatly arranged on a white porcelain platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-136",
    "title": "🥩 萝卜烧牛肉",
    "cleanTitle": "萝卜烧牛肉",
    "prompt": "A minimalist plated dish of radish braised beef. Beef brisket cubes, white daikon, and chestnuts coated in dark savory glaze, neatly arranged in a white ceramic bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-137",
    "title": "🥩 青椒炒猪肝",
    "cleanTitle": "青椒炒猪肝",
    "prompt": "A minimalist plated dish of pepper pork liver. Pork liver and green bell peppers coated in savory brown soy glaze, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-138",
    "title": "🥚 培根滑蛋西蓝花",
    "cleanTitle": "培根滑蛋西蓝花",
    "prompt": "A minimalist plated dish of bacon egg broccoli. Scrambled eggs, smoked bacon, and broccoli coated in light aromatic butter, neatly arranged on a white ceramic platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-139",
    "title": "🥣 茄汁莜面窝窝",
    "cleanTitle": "茄汁莜面窝窝",
    "prompt": "A minimalist plated dish of oat noodle rolls. Hollow oat noodle rolls stood upright coated in red tomato celery gravy, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-140",
    "title": "🥔 红薯糙米饭",
    "cleanTitle": "红薯糙米饭",
    "prompt": "A minimalist plated dish of sweet potato rice. Steamed brown rice and sweet potato cubes coated in toasted sesame sheen, neatly arranged in a white ceramic bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-141",
    "title": "🐟 芹菜鱼丝",
    "cleanTitle": "芹菜鱼丝",
    "prompt": "A minimalist plated dish of celery fish shreds. White fish strips and emerald celery batons coated in light wine glaze, neatly arranged on a white porcelain plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-142",
    "title": "🧈 牡蛎豆腐羹",
    "cleanTitle": "牡蛎豆腐羹",
    "prompt": "A minimalist plated dish of oyster tofu soup. Fresh oysters and silken tofu slices coated in savory seafood broth, neatly arranged in a white porcelain bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-143",
    "title": "🦐 蒜香海带丝",
    "cleanTitle": "蒜香海带丝",
    "prompt": "A minimalist plated dish of garlic kelp salad. Shredded kelp ribbons and minced garlic coated in black vinegar dressing, neatly arranged in a white porcelain bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-144",
    "title": "🦐 紫菜虾皮蛋花汤",
    "cleanTitle": "紫菜虾皮蛋花汤",
    "prompt": "A minimalist plated dish of seaweed egg soup. Purple seaweed, dried shrimp, and egg ribbons coated in clear fragrant broth, neatly arranged in a white porcelain bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-145",
    "title": "🥩 玲珑牛奶馒头",
    "cleanTitle": "玲珑牛奶馒头",
    "prompt": "A minimalist plated dish of steamed milk mantou. Steamed white milk mantou buns coated in glossy sugar sheen, neatly arranged on a white porcelain platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 48
  },
  {
    "id": "cn-146",
    "title": "🍗 小葱炒鸡蛋",
    "cleanTitle": "小葱炒鸡蛋",
    "prompt": "A minimalist plated dish of scallion eggs. Fluffy golden egg curds and green scallions coated in fragrant cooking oil, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "cn-147",
    "title": "🍳 松仁玉米",
    "cleanTitle": "松仁玉米",
    "prompt": "A minimalist plated dish of pine nut corn. Golden corn kernels and roasted pine nuts coated in light vegetable oil, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-148",
    "title": "🥢 红枣芝麻红豆豆浆",
    "cleanTitle": "红枣芝麻红豆豆浆",
    "prompt": "A minimalist plated dish of sesame red bean drink. Red bean and black sesame soy milk coated in velvety microfoam, neatly arranged in a white ceramic mug. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-149",
    "title": "🥩 菠菜炒猪肝",
    "cleanTitle": "菠菜炒猪肝",
    "prompt": "A minimalist plated dish of spinach pork liver. Sliced pork liver and emerald spinach leaves coated in garlic wine sauce, neatly arranged on a white ceramic platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-150",
    "title": "🥣 小窝窝头",
    "cleanTitle": "小窝窝头",
    "prompt": "A minimalist plated dish of steamed cornmeal buns. Cone-shaped cornmeal wotou buns with hollow bases coated in rustic grain sheen, neatly arranged on a white porcelain platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "cn-151",
    "title": "🥩 胡萝卜炒牛肉丝",
    "cleanTitle": "胡萝卜炒牛肉丝",
    "prompt": "A minimalist plated dish of carrot beef shreds. Beef strips and julienned carrot threads coated in light savory soy sauce, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "hsh-01-hazelnut-mocha",
    "title": "🏆 榛果摩卡特饮干粉 (Hazelnut Mocha Mix)",
    "cleanTitle": "榛果摩卡特饮干粉 (Hazelnut Mocha Mix)",
    "prompt": "A minimalist plated dish of hazelnut mocha. Frothy espresso mocha beverage dusted with cocoa coated in chocolate drizzle, neatly arranged in a white ceramic mug. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 48
  },
  {
    "id": "hsh-02-taco-soup",
    "title": "🏆 塔可墨西哥风味浓汤 (Taco Soup)",
    "cleanTitle": "塔可墨西哥风味浓汤 (Taco Soup)",
    "prompt": "A minimalist plated dish of taco soup. Ground turkey, pinto beans, and sweet corn kernels coated in spiced tomato broth, neatly arranged in a white porcelain bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "hsh-03-chicken-ritz",
    "title": "🏆 Ritz饼干金黄烤鸡 (Chicken Ritz)",
    "cleanTitle": "Ritz饼干金黄烤鸡 (Chicken Ritz)",
    "prompt": "A minimalist plated dish of Ritz chicken. Tender baked chicken breast and sweet peas coated in golden buttery cracker crust, neatly arranged in a white ceramic dish. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "hsh-04-mexican-lasagna",
    "title": "🏆 墨西哥风味千层饼 (Mexican Lasagna)",
    "cleanTitle": "墨西哥风味千层饼 (Mexican Lasagna)",
    "prompt": "A minimalist plated dish of Mexican lasagna. Layered ground beef, pasta ribbons, and salsa coated in melted cheddar and mozzarella, neatly arranged on a white ceramic platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "hsh-05-noodle-kugel",
    "title": "🏆 祖母秘制面条库格尔 (Grandma’s Noodle Kugel)",
    "cleanTitle": "祖母秘制面条库格尔 (Grandma’s Noodle Kugel)",
    "prompt": "A minimalist plated dish of baked noodle kugel. Baked egg noodles and cream cheese custard coated in cinnamon sugar glaze, neatly arranged on a white porcelain plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "hsh-06-bbq-butter-beans",
    "title": "🏆 烧烤黄油豆 (Barbecued Butter Beans)",
    "cleanTitle": "烧烤黄油豆 (Barbecued Butter Beans)",
    "prompt": "A minimalist plated dish of barbecue butter beans. Butter beans and crispy caramelized bacon coated in smoky barbecue sauce, neatly arranged in a white ceramic bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "hsh-07-pineapple-stuffing",
    "title": "🏆 菠萝香面包填料 (Pineapple Stuffing)",
    "cleanTitle": "菠萝香面包填料 (Pineapple Stuffing)",
    "prompt": "A minimalist plated dish of pineapple stuffing. Toasted bread cubes and sweet crushed pineapple coated in buttery egg custard glaze, neatly arranged in a white ceramic dish. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "hsh-08-pecan-rice",
    "title": "🏆 碧根果果仁米饭 (Pecan Rice)",
    "cleanTitle": "碧根果果仁米饭 (Pecan Rice)",
    "prompt": "A minimalist plated dish of pecan rice. Fluffy rice pilaf and toasted brown pecans coated in fragrant melted butter, neatly arranged on a white porcelain plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "hsh-09-peanut-butter-brownie",
    "title": "🏆 酥脆花生酱大理石布朗尼 (Crunchy PB Brownies)",
    "cleanTitle": "酥脆花生酱大理石布朗尼 (Crunchy PB Brownies)",
    "prompt": "A minimalist plated dish of peanut butter brownies. Brownie bars swirled with peanut butter coated in glossy chocolate glaze, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "hsh-10-italian-mushrooms",
    "title": "👨‍🍳 主厨意式酿烤蘑菇 (Italian Mushrooms)",
    "cleanTitle": "主厨意式酿烤蘑菇 (Italian Mushrooms)",
    "prompt": "A minimalist plated dish of Italian stuffed mushrooms. Roasted mushroom caps filled with melted mozzarella coated in herb garlic butter, neatly arranged on a white ceramic platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "hsh-11-pecan-rolls",
    "title": "🥐 过夜美洲山核桃肉桂卷 (Overnight Pecan Rolls)",
    "cleanTitle": "过夜美洲山核桃肉桂卷 (Overnight Pecan Rolls)",
    "prompt": "A minimalist plated dish of caramel pecan rolls. Baked cinnamon rolls inverted with sticky caramel coated in toasted pecan halves, neatly arranged on a white porcelain plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "hsh-12-pound-cake",
    "title": "🍰 老式酸奶油磅蛋糕 (Sour Cream Pound Cake)",
    "cleanTitle": "老式酸奶油磅蛋糕 (Sour Cream Pound Cake)",
    "prompt": "A minimalist plated dish of sour cream pound cake. Pound cake slices showing dense crumb coated in powdered sugar dusting, neatly arranged on a white porcelain plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "hsh-13-beef-barley-soup",
    "title": "🍲 牛肉大麦蔬菜浓汤 (Beef & Barley Vegetable Soup)",
    "cleanTitle": "牛肉大麦蔬菜浓汤 (Beef & Barley Vegetable Soup)",
    "prompt": "A minimalist plated dish of beef barley soup. Ground beef chunks and pearl barley grains coated in savory tomato broth, neatly arranged in a white porcelain bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "hsh-14-angel-biscuits",
    "title": "🥐 祖母天使比司吉饼干 (Angel Biscuits)",
    "cleanTitle": "祖母天使比司吉饼干 (Angel Biscuits)",
    "prompt": "A minimalist plated dish of golden angel biscuits. Flaky buttery yeast biscuits split open coated in golden crust sheen, neatly arranged on a white porcelain platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "hsh-15-chicken-dumplings",
    "title": "🍲 家常美式鸡肉炖面团 (Chicken & Dumplings)",
    "cleanTitle": "家常美式鸡肉炖面团 (Chicken & Dumplings)",
    "prompt": "A minimalist plated dish of chicken and dumplings. Tender chicken and fluffy steamed dumplings coated in creamy veloute broth, neatly arranged in a white porcelain bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "hsh-16-hashbrown-casserole",
    "title": "🥔 橄榄球硬汉土豆饼芝士焗煲 (Hashbrown Casserole)",
    "cleanTitle": "橄榄球硬汉土豆饼芝士焗煲 (Hashbrown Casserole)",
    "prompt": "A minimalist plated dish of hashbrown casserole. Crispy shredded potato hashbrowns and cheddar coated in rich sour cream glaze, neatly arranged in a white ceramic baking dish. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "v3-espresso-brownies",
    "title": "Espresso Brownies 意式浓缩布朗尼",
    "cleanTitle": "Espresso Brownies 意式浓缩布朗尼",
    "prompt": "A minimalist plated dish of espresso brownies. Dark chocolate brownie squares with cracked crust coated in fudgy espresso glaze, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "v3-hong-shao-rou",
    "title": "毛氏红烧肉",
    "cleanTitle": "毛氏红烧肉",
    "prompt": "A minimalist plated dish of Mao braised pork. Pork belly cubes with alternating fat-lean layers coated in amber caramel glaze, neatly arranged in a white porcelain bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "v3-caesar-salad",
    "title": "Classic Caesar Salad 经典凯撒沙拉",
    "cleanTitle": "Classic Caesar Salad 经典凯撒沙拉",
    "prompt": "A minimalist plated dish of classic Caesar salad. Romaine lettuce hearts and toasted garlic croutons coated in creamy Caesar dressing, neatly arranged in a white porcelain bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  }
]

SITUATION_DATA = [
  {
    "id": "situation-stirfry",
    "title": "🔥 时令快炒 (Wok Stir-Fry)",
    "cleanTitle": "时令快炒 (Wok Stir-Fry)",
    "prompt": "A minimalist plated dish of wok stir-fry. Crisp emerald snow peas and tender meat slices coated in glistening garlic glaze, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "situation-stew",
    "title": "🍲 滋补慢炖 (Stew & Braise)",
    "cleanTitle": "滋补慢炖 (Stew & Braise)",
    "prompt": "A minimalist plated dish of braised stew. Tender seared beef cubes and glazed carrot chunks coated in rich amber gravy, neatly arranged in a white porcelain casserole. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "situation-appetizer",
    "title": "🥗 精致前菜 (Cold Platter & Appetizer)",
    "cleanTitle": "精致前菜 (Cold Platter & Appetizer)",
    "prompt": "A minimalist plated dish of chilled appetizer. Crisp julienned cucumber ribbons and marinated tofu coated in fragrant sesame chili oil, neatly arranged on a white ceramic platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "situation-main",
    "title": "🥩 主厨大件主菜 (Signature Main Course)",
    "cleanTitle": "主厨大件主菜 (Signature Main Course)",
    "prompt": "A minimalist plated dish of prime roast. Seared beef medallion showing rich marbled grain coated in glossy peppercorn jus, neatly arranged on a white porcelain platter. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "situation-soup",
    "title": "🥣 清润靓汤 (Nourishing Soup & Broth)",
    "cleanTitle": "清润靓汤 (Nourishing Soup & Broth)",
    "prompt": "A minimalist plated dish of nourishing broth. Tender slow-cooked rib chunks and sweet corn rounds coated in clear golden broth, neatly arranged in a white porcelain tureen. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "situation-salad",
    "title": "🥗 生鲜沙拉 (Fresh Garden Salad)",
    "cleanTitle": "生鲜沙拉 (Fresh Garden Salad)",
    "prompt": "A minimalist plated dish of garden salad. Crisp vibrant romaine hearts and cherry tomatoes coated in light citrus dressing, neatly arranged in a white porcelain bowl. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  },
  {
    "id": "situation-snack",
    "title": "🥟 休闲小吃点心 (Artisanal Snack & Dim Sum)",
    "cleanTitle": "休闲小吃点心 (Artisanal Snack & Dim Sum)",
    "prompt": "A minimalist plated dish of artisanal dumplings. Translucent crystal dumplings filled with plump pink shrimp coated in light scallion oil, neatly arranged on a white ceramic plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 50
  },
  {
    "id": "situation-dessert",
    "title": "🍰 烘焙甜品 (Pastry & Dessert)",
    "cleanTitle": "烘焙甜品 (Pastry & Dessert)",
    "prompt": "A minimalist plated dish of gourmet pastry. Dark chocolate tart with golden butter crust coated in glossy cocoa glaze, neatly arranged on a white porcelain plate. Shot on Fujifilm GFX 100S, 110mm f/2. Dramatic warm side lighting, shallow depth of field, appetizing glossy texture, Kodak Gold 200 color grading.",
    "wordCount": 49
  }
]

print(f"✅ 成功加载 {len(RECIPES_DATA)} 道食谱指令与 {len(SITUATION_DATA)} 个通用情境保底指令！")
print("食谱首道样例:", RECIPES_DATA[0]['id'], RECIPES_DATA[0]['cleanTitle'])
print("  指令:", RECIPES_DATA[0]['prompt'])
print("  词数:", RECIPES_DATA[0]['wordCount'], "词")
print("情境首道样例:", SITUATION_DATA[0]['id'], SITUATION_DATA[0]['cleanTitle'])
print("  指令:", SITUATION_DATA[0]['prompt'])
print("  词数:", SITUATION_DATA[0]['wordCount'], "词")


In [ ]:
# [Cell 5] 4:3 居中裁切 + ≤30KB WebP 自适应压制 + HTML Base64 原生渲染引擎
def post_process_cover(raw_img: Image.Image, max_kb: int = 30, target_width: int = 1200, target_height: int = 900) -> tuple[bytes, float, int]:
    """
    1. 中心裁切为 4:3 构图
    2. Lanczos 高保真重采样
    3. 阶梯式递减 quality 进行 WebP 压缩，确保单张严格 <= max_kb (30KB)
    """
    w, h = raw_img.size
    target_ratio = 4.0 / 3.0
    current_ratio = w / h
    
    if current_ratio > target_ratio:
        new_w = int(h * target_ratio)
        left = (w - new_w) // 2
        cropped = raw_img.crop((left, 0, left + new_w, h))
    else:
        new_h = int(w / target_ratio)
        top = (h - new_h) // 2
        cropped = raw_img.crop((0, top, w, top + new_h))
        
    resized = cropped.resize((target_width, target_height), Image.Resampling.LANCZOS)
    
    quality = 80
    buf = io.BytesIO()
    resized.save(buf, format='WEBP', quality=quality, method=6)
    size_kb = len(buf.getvalue()) / 1024.0
    
    # 阶梯式降低质量直到符合体积要求
    while size_kb > max_kb and quality > 35:
        quality -= 5
        buf = io.BytesIO()
        resized.save(buf, format='WEBP', quality=quality, method=6)
        size_kb = len(buf.getvalue()) / 1024.0
        
    # 若极端复杂场景仍超标，降至 800x600 严格保底
    if size_kb > max_kb:
        sub_resized = cropped.resize((800, 600), Image.Resampling.LANCZOS)
        buf = io.BytesIO()
        sub_resized.save(buf, format='WEBP', quality=75, method=6)
        size_kb = len(buf.getvalue()) / 1024.0
        
    return buf.getvalue(), round(size_kb, 2), quality


def render_preview_card(webp_bytes: bytes, title: str, subtitle: str = "", width: int = 320):
    """
    完全替代容易报错崩溃的 IPImage(..., format='webp')
    使用浏览器原生 HTML Base64 内嵌，零异常、高兼容
    """
    b64 = base64.b64encode(webp_bytes).decode('utf-8')
    sub_html = f"<div style='font-size:12px;color:#64748b;margin-top:4px;'>{subtitle}</div>" if subtitle else ""
    html = f"""
    <div style="display:inline-block;margin:6px;padding:10px;background:#ffffff;border:1px solid #e2e8f0;border-radius:10px;box-shadow:0 2px 5px rgba(0,0,0,0.06);font-family:sans-serif;">
        <div style="font-weight:600;font-size:13px;color:#0f172a;margin-bottom:6px;">{title}</div>
        <img src="data:image/webp;base64,{b64}" width="{width}" style="border-radius:6px;display:block;" />
        {sub_html}
    </div>
    """
    display(HTML(html))


def cleanup_vram():
    """显存与垃圾回收守卫，确保 170 道连续批处理永不 OOM"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("✅ 图像自适应处理、HTML Base64 原生渲染与显存守卫就绪！")

In [ ]:
#@title 🚀 [Cell 6] 端到端全量自动化批处理流水线 (支持全量/分段/断点续跑/4步出图)
#@markdown 选择本次运行范围（可选全量 178 项、仅跑 8 个通用情境保底图、或仅跑 170 道食谱）：
PROCESS_SCOPE = "ALL" #@param ["ALL", "SITUATIONS_ONLY", "RECIPES_ONLY"]
FORCE_OVERWRITE = False #@param {type:"boolean"}
SHOW_PREVIEW = True #@param {type:"boolean"}
TARGET_MAX_KB = 30 #@param {type:"integer"}

# 读取本地 Drive Manifest
manifest = {}
if os.path.exists(MANIFEST_PATH):
    try:
        with open(MANIFEST_PATH, 'r', encoding='utf-8') as f:
            manifest = json.load(f)
        print(f"📂 成功加载历史进度清单，已有 {len(manifest)} 项记录。")
    except Exception:
        manifest = {}

if PROCESS_SCOPE == "SITUATIONS_ONLY":
    TASK_LIST = SITUATION_DATA
elif PROCESS_SCOPE == "RECIPES_ONLY":
    TASK_LIST = RECIPES_DATA
else:
    TASK_LIST = SITUATION_DATA + RECIPES_DATA

total = len(TASK_LIST)
success_count = 0
skip_count = 0
fail_count = 0
t_start_all = time.time()

print(f"\n🔥 流水线启动！当前任务队列: {total} 个目标 | 模式: {PROCESS_SCOPE} | 模型: FLUX.1-schnell (4步推断) | 显存守卫: 已激活\n")

for idx, item in enumerate(TASK_LIST, start=1):
    rid = item['id']
    title = item['cleanTitle']
    file_name = f"{rid}.webp"
    cdn_url = f"{SUPABASE_URL}/storage/v1/object/public/{BUCKET_NAME}/{file_name}"
    
    # 1. 云端与 Manifest 双重断点检测
    is_in_cloud = file_name in CLOUD_EXISTING_FILES
    is_in_manifest = (rid in manifest and manifest[rid].get('status') == 'success')
    
    if not FORCE_OVERWRITE and (is_in_cloud or is_in_manifest):
        skip_count += 1
        if skip_count % 15 == 0 or idx == total:
            print(f"[{idx}/{total}] ⏩ 已跳过已存在封面: {rid} {title}")
        continue
        
    print(f"[{idx}/{total}] 🎨 正在生成: [{rid}] {title}...")
    t0 = time.time()
    
    try:
        # 2. FLUX.1-schnell 4 步极速推断
        with torch.inference_mode():
            raw_img = pipe(
                prompt=item['prompt'],
                num_inference_steps=4,
                max_sequence_length=256,
                width=1024,
                height=1024
            ).images[0]
            
        gen_time = round(time.time() - t0, 1)
        
        # 3. 4:3 裁切与 WebP 自适应压制 (≤ 30KB)
        webp_bytes, kb_size, final_q = post_process_cover(raw_img, max_kb=TARGET_MAX_KB)
        
        # 4. 上传至 Supabase Storage
        supabase.storage.from_(BUCKET_NAME).upload(
            path=file_name,
            file=webp_bytes,
            file_options={"content-type": "image/webp", "upsert": "true"}
        )
        CLOUD_EXISTING_FILES.add(file_name)
        
        # 5. 持久化记录至 Drive Manifest
        manifest[rid] = {
            "id": rid,
            "title": title,
            "cdnUrl": cdn_url,
            "sizeKb": kb_size,
            "quality": final_q,
            "model": "FLUX.1-schnell",
            "status": "success",
            "updatedAt": time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())
        }
        with open(MANIFEST_PATH, 'w', encoding='utf-8') as f:
            json.dump(manifest, f, ensure_ascii=False, indent=2)
            
        print(f"   ✨ 完成! 耗时: {gen_time}s | 体积: {kb_size}KB (q={final_q}) | CDN: {cdn_url}")
        if SHOW_PREVIEW:
            render_preview_card(webp_bytes, f"[{rid}] {title}", f"{kb_size} KB | {gen_time}s")
            
        success_count += 1
        
    except Exception as e:
        fail_count += 1
        print(f"   ❌ 处理异常: {rid} {title} -> {e}")
        manifest[rid] = {"id": rid, "title": title, "status": "failed", "error": str(e)}
        with open(MANIFEST_PATH, 'w', encoding='utf-8') as f:
            json.dump(manifest, f, ensure_ascii=False, indent=2)
            
    finally:
        # 6. 强制显存与内存回收，保障流水线长期稳定
        if 'raw_img' in locals():
            del raw_img
        if 'webp_bytes' in locals():
            del webp_bytes
        cleanup_vram()

total_min = round((time.time() - t_start_all) / 60, 2)
print("\n" + "=" * 75)
print(f"🏁 批处理结束！总耗时: {total_min} 分钟")
print(f"📊 本次新生成: {success_count} 项 | 历史跳过: {skip_count} 项 | 异常失败: {fail_count} 项")
print(f"💾 最新清单已持久化备份至 Google Drive: {MANIFEST_PATH}")

In [ ]:
# [Cell 7] 最终自检审计与成果抽样报告
if os.path.exists(MANIFEST_PATH):
    with open(MANIFEST_PATH, 'r', encoding='utf-8') as f:
        mf = json.load(f)
        
    valid_items = [v for v in mf.values() if v.get('status') == 'success']
    sizes = [v['sizeKb'] for v in valid_items if 'sizeKb' in v]
    total_all = len(RECIPES_DATA) + len(SITUATION_DATA)
    
    print("📋 最终达标自检结果:")
    print(f"   - 覆盖进度: {len(valid_items)} / {total_all} ({round(len(valid_items)/total_all*100, 1)}%)")
    if sizes:
        print(f"   - 平均文件体积: {round(sum(sizes)/len(sizes), 2)} KB (严格全部 ≤ 30KB)")
        print(f"   - 最大体积: {max(sizes)} KB | 最小体积: {min(sizes)} KB")
        
    print("\n🖼 最新入库 3 张图像公网 CDN 抽检展示:")
    for s in valid_items[-3:]:
        print(f"🍽 [{s['id']}] {s['title']} ({s['sizeKb']} KB) -> {s['cdnUrl']}")
        try:
            display(HTML(f'<img src="{s["cdnUrl"]}" width="280" style="border-radius:8px;margin:6px;" />'))
        except Exception:
            pass
else:
    print("当前尚未生成 Manifest 清单。")